### <span style="color:lightgreen"> Stock Price Prediction using RNN </span>

In [13]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

# Load dataset
file_path = "Equity-SHRIRAMFIN-EQ.csv"
data = pd.read_csv(file_path)

#print(data)

# Clean column names (remove spaces, make lowercase)
data.columns = data.columns.str.strip().str.lower()

# Convert numerical columns to float (handle commas)
for col in ['open', 'high', 'low', 'close']:
    data[col] = data[col].astype(str).str.replace(',', '').astype(float)

# Convert 'date' column to datetime format
data['date'] = pd.to_datetime(data['date'])

# Drop any missing values
data = data.dropna()

# Select features (Date, Open, High, Low, Close)
features = ['open', 'high', 'low', 'close']
data_selected = data[features]
dates = data['date'].values  # Store dates separately

# Normalize features using MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data_selected)

# Function to create time-windowed dataset
def create_dataset(data, dates, time_window):
    X, y, date_list = [], [], []
    for i in range(len(data) - time_window):
        X.append(data[i:i + time_window])
        y.append(data[i + time_window, -1])  # Predicting the Close price
        date_list.append(dates[i + time_window])  # Store corresponding date
    return np.array(X), np.array(y), np.array(date_list)

# Define time window for sequence prediction
time_window = 8
X, y, y_dates = create_dataset(data_scaled, dates, time_window)

# Reshape X for RNN input (samples, time_steps, features)
X = X.reshape(X.shape[0], X.shape[1], len(features))

# Train-test split (80% train, 20% test)
split = int(0.8 * len(X))
if split == 0 or split >= len(X):
    print("Error: Not enough data for proper train-test split.")
    exit()

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
y_test_dates = y_dates[split:]  # Get corresponding dates for test set

# Build the RNN model
model = Sequential([
    SimpleRNN(10, activation='tanh', input_shape=(time_window, len(features))),
    Dense(1)
])

# Compile model
model.compile(optimizer='adam', loss='mse')
model.summary()

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=10, validation_data=(X_test, y_test))

# Predict on test data
predictions = model.predict(X_test)

# Inverse transform predictions and actual values
close_index = features.index('close')  # Get index of 'close' in features
y_test_original = scaler.inverse_transform(
    np.column_stack([X_test[:, -1, :-1], y_test]))[:, close_index]
predictions_original = scaler.inverse_transform(
    np.column_stack([X_test[:, -1, :-1], predictions]))[:, close_index]

# Ensure enough values to print
num_values = min(len(y_test_original), len(predictions_original), len(y_test_dates))

print("\nDate-wise Actual vs Predicted Stock Prices:")
for i in range(num_values):
    print(f"Date: {pd.Timestamp(y_test_dates[i]).date()} | Actual: {y_test_original[i]:.2f} | Predicted: {predictions_original[i]:.2f}")


# ------------------ Predict Next 2 Days ------------------ #
# Get the last 'time_window' days from the dataset
last_known_data = data_scaled[-time_window:]

future_predictions = []
future_dates = pd.date_range(start=pd.Timestamp(dates[-1]) + pd.Timedelta(days=1), periods=2, freq='B')  # Next 2 business days

# Predict iteratively for the next 2 days
for i in range(2):
    last_known_data_reshaped = last_known_data.reshape(1, time_window, len(features))  # Reshape for RNN input
    next_pred_scaled = model.predict(last_known_data_reshaped)[0, 0]  # Get the predicted 'close' price

    # Create a new feature vector for the next day (assume other features remain the same as the last known values)
    next_day_scaled = np.append(last_known_data[1:], [[last_known_data[-1, 0], last_known_data[-1, 1], last_known_data[-1, 2], next_pred_scaled]], axis=0)

    # Store prediction
    future_predictions.append(next_pred_scaled)

    # Update last_known_data for next iteration
    last_known_data = next_day_scaled

# Print actual vs predicted for test data
print("\nDate-wise Actual vs Predicted Stock Prices:")
for i in range(len(y_test_original)):
    print(f"Date: {y_test_dates[i]} | Actual: {y_test_original[i]:.2f} | Predicted: {predictions_original[i]:.2f}")

# Convert predicted scaled values back to original price scale
future_predictions_original = [
    scaler.inverse_transform([[last_known_data[-1, 0], last_known_data[-1, 1], last_known_data[-1, 2], pred]])[0, close_index]
    for pred in future_predictions
]

# Print next 2 days predictions with new dates
print("\nPredicted Prices for Next 2 Days:")
for i in range(1):
    print(f"Date: {future_dates[i]} | Predicted Close Price: {future_predictions_original[i]:.2f}")




/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_6 (SimpleRNN)        │ (None, 10)             │           150 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 161 (644.00 B)

 Trainable params: 161 (644.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 688ms/step - loss: 0.0983 - val_loss: 0.3084
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.0889 - val_loss: 0.2889
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.0805 - val_loss: 0.2698
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.0730 - val_loss: 0.2511
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.0665 - val_loss: 0.2331
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.0607 - val_loss: 0.2160
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.0558 - val_loss: 0.1998
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.0515 - val_loss: 0.1847
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.0479 - val_loss: 0.1707
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.0447 - val_loss: 0.1579
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.0419 - val_loss: 0.1462
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.0395 - val_loss: 0.1357
